<a href="https://colab.research.google.com/github/yudithvega-art/bhm-mrsa-indonesia/blob/main/Run04_BHM15072026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import argparse
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore")

SECTION_ROWS = {
    "Milk": (4, 23), "Bioaerosol": (24, 53),
    "Env_Dust_Swab": (54, 124), "Water": (125, 158),
}
SECTION_TO_MATRIX = {
    "Milk": "Milk", "Bioaerosol": "Bioaerosol",
    "Env_Dust_Swab": "EnvDust", "Water": "Water",
}
MATRIX_ORDER = ["Milk", "Water", "Bioaerosol", "EnvDust"]
MATRIX_UNITS = {"Milk": "CFU/mL", "Water": "CFU/mL",
                "Bioaerosol": "CFU/m3", "EnvDust": "CFU/cm2"}
MATRIX_COLORS = {"Milk": "#D55E00", "Water": "#0072B2",
                 "Bioaerosol": "#009E73", "EnvDust": "#CC79A7"}

# Prior on grand mean eta (log10 scale), matrix-specific centering per Methods 2.3
PRIOR_ETA = {
    "Milk": (2.5, 1.0), "Water": (0.5, 1.5),
    "EnvDust": (1.5, 1.5), "Bioaerosol": (3.0, 1.0),
}

LOD_LOG10 = 0.0    # log10(1 CFU) = 0 ; values at/below treated as censored
COL_CODE, COL_CFU, COL_LOG10 = 2, 3, 4

mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 160, "savefig.bbox": "tight",
})


def to_F(p):
    return f"F-{int(p):02d}"


def parse_point(code, matrix):
    parts = str(code).split(".")
    if matrix == "Bioaerosol":
        for p in parts:
            if re.match(r"^T\d", p):
                return p
    if matrix == "Water":
        for p in parts:
            if p in ("Inlet", "Outlet", "T1", "T2", "T3"):
                return p
        return "other"
    if matrix == "EnvDust":
        return parts[1] if len(parts) >= 2 else "NA"
    return "udder"


def load_data(input_xlsx):
    """Return {matrix: DataFrame[farm, point, log10, status]}.
    status in {'obs','cens','zero'}."""
    df = pd.read_excel(input_xlsx, sheet_name="isolates", header=None)
    out = {}
    for section, (start, end) in SECTION_ROWS.items():
        matrix = SECTION_TO_MATRIX[section]
        sub = df.iloc[start:end]
        rows = []
        for _, r in sub.iterrows():
            code = str(r[COL_CODE])
            m = re.match(r"^P(\d+)", code)
            if not m:
                continue
            farm = to_F(m.group(1))
            point = f"{farm}|{parse_point(code, matrix)}"
            cfu = pd.to_numeric(r[COL_CFU], errors="coerce")
            log10 = pd.to_numeric(r[COL_LOG10], errors="coerce")
            if pd.isna(cfu):
                continue
            if cfu == 0:
                rows.append({"farm": farm, "point": point,
                             "log10": np.nan, "status": "zero"})
            elif cfu <= 1:      # 0 < CFU <= 1  -> left-censored at LOD
                rows.append({"farm": farm, "point": point,
                             "log10": LOD_LOG10, "status": "cens"})
            else:               # CFU > 1  -> observed
                rows.append({"farm": farm, "point": point,
                             "log10": float(log10), "status": "obs"})
        out[matrix] = pd.DataFrame(rows)
    return out


def fit_full_model(data, matrix, draws, tune, chains, seed=42):
    """Fit 3-level zero-inflated lognormal with left-censoring."""
    import pymc as pm
    import pytensor.tensor as pt

    farms = sorted(data["farm"].unique(), key=lambda f: int(f.split("-")[1]))
    farm_idx = {f: i for i, f in enumerate(farms)}
    points = sorted(data["point"].unique())
    point_idx = {p: i for i, p in enumerate(points)}
    point_farm = np.array([farm_idx[p.split("|")[0]] for p in points])

    obs = data[data["status"] == "obs"]
    cens = data[data["status"] == "cens"]

    y_obs = obs["log10"].values
    obs_pt = obs["point"].map(point_idx).values.astype(int)
    cens_pt = cens["point"].map(point_idx).values.astype(int)
    n_cens = len(cens)

    prior_mean, prior_sd = PRIOR_ETA[matrix]

    with pm.Model() as model:
        eta = pm.Normal("eta", mu=prior_mean, sigma=prior_sd)
        tau = pm.HalfNormal("tau", sigma=1.0)
        omega = pm.HalfNormal("omega", sigma=1.0)
        sigma = pm.HalfNormal("sigma", sigma=1.0)

        # non-centered farm level
        z_farm = pm.Normal("z_farm", 0.0, 1.0, shape=len(farms))
        theta = pm.Deterministic("theta", eta + tau * z_farm)

        # non-centered point level
        z_pt = pm.Normal("z_pt", 0.0, 1.0, shape=len(points))
        mu_pt = pm.Deterministic("mu_pt", theta[point_farm] + omega * z_pt)

        # observed concentrations
        pm.Normal("obs", mu=mu_pt[obs_pt], sigma=sigma, observed=y_obs)

        # left-censored contributions: P(y <= LOD) via normal CDF
        if n_cens > 0:
            cens_mu = mu_pt[cens_pt]
            logcdf = pm.logcdf(pm.Normal.dist(mu=cens_mu, sigma=sigma),
                               LOD_LOG10)
            pm.Potential("cens", pt.sum(logcdf))

        idata = pm.sample(draws=draws, tune=tune, chains=chains, cores=1,
                          target_accept=0.99, random_seed=seed,
                          progressbar=False)
    return idata, farms


def summarize(idata, farms, matrix):
    import arviz as az
    post = idata.posterior
    theta = post["theta"].stack(s=("chain", "draw")).values
    eta = post["eta"].stack(s=("chain", "draw")).values

    rows = []
    for j, f in enumerate(farms):
        s = theta[j]
        rows.append({"matrix": matrix, "farm": f,
                     "median": float(np.median(s)),
                     "q025": float(np.percentile(s, 2.5)),
                     "q975": float(np.percentile(s, 97.5))})
    mu_df = pd.DataFrame(rows)

    rhat = az.rhat(idata)
    max_rhat = float(max(float(rhat["eta"].values),
                         float(rhat["tau"].values),
                         float(rhat["omega"].values),
                         float(rhat["sigma"].values),
                         float(rhat["theta"].max().values)))
    ess = az.ess(idata)
    min_ess = float(min(float(ess["eta"].values),
                        float(ess["theta"].min().values)))
    n_div = int(idata.sample_stats["diverging"].sum())

    gm = {"matrix": matrix, "unit": MATRIX_UNITS[matrix], "n_farms": len(farms),
          "eta_log10_median": float(np.median(eta)),
          "eta_log10_q025": float(np.percentile(eta, 2.5)),
          "eta_log10_q975": float(np.percentile(eta, 97.5)),
          "C_linear_median": 10 ** float(np.median(eta)),
          "C_linear_q025": 10 ** float(np.percentile(eta, 2.5)),
          "C_linear_q975": 10 ** float(np.percentile(eta, 97.5)),
          "max_rhat": round(max_rhat, 4),
          "min_ess": int(min_ess),
          "divergences": n_div,
          "total_draws": theta.shape[1]}
    return mu_df, gm


def plot_forest(mu_all, grand, outpath):
    fig, axes = plt.subplots(1, 4, figsize=(17, 6.5))
    for i, (ax, matrix) in enumerate(zip(axes, MATRIX_ORDER)): # Added enumerate
        sub_matrix = mu_all[mu_all["matrix"] == matrix]
        # Filter for farms F-01 to F-10 and sort numerically by farm ID
        # Corrected filtering logic to return a boolean series
        sub_filtered = sub_matrix[sub_matrix['farm'].apply(lambda x: re.match(r'^F-(0[1-9]|10)$', x) is not None)].copy()
        sub_filtered['farm_num'] = sub_filtered['farm'].apply(lambda x: int(x.split('-')[1]))
        sub_to_display = sub_filtered.sort_values('farm_num').drop(columns=['farm_num'])

        farms_to_display = sub_to_display["farm"].tolist()

        y = np.arange(len(farms_to_display))
        color = MATRIX_COLORS[matrix]
        central = 10 ** sub_to_display["median"].values
        lo = np.maximum(10 ** sub_to_display["q025"].values, 1e-6)
        hi = 10 ** sub_to_display["q975"].values
        ax.errorbar(central, y, xerr=[central - lo, hi - central],
                    fmt="o", color=color, ecolor=color, alpha=0.85,
                    markersize=9, capsize=4, linewidth=1.6,
                    markeredgecolor="white", markeredgewidth=1.3)
        ax.set_yticks(y);
        ax.set_yticklabels(farms_to_display, fontsize=10)
        ax.set_xscale("log")
        ax.set_xlabel(f"Concentration ({MATRIX_UNITS[matrix]})  ·  log scale")
        # Changed subtitle to A, B, C... and removed 'n = ... farms'
        ax.set_title(f"{chr(65 + i)}", loc="left", pad=8)
        ax.grid(axis="x", alpha=0.25, which="major")
        ax.grid(axis="x", alpha=0.10, which="minor", linestyle=":")
        ax.invert_yaxis()
        g = grand[grand["matrix"] == matrix]
        if not g.empty:
            ax.axvline(10 ** g["eta_log10_median"].values[0], ls=":",
                       color="gray", alpha=0.7, lw=1.1)
    # Removed suptitle entirely
    # plt.suptitle("BHM posterior MRSA concentrations per farm and matrix  "
    #              "(3-level zero-inflated model, left-censored, median + 95% CrI)",
    #              y=1.01)
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


def main(input_xlsx, output_dir, draws, tune, chains):
    out = Path(output_dir); out.mkdir(exist_ok=True, parents=True)
    data = load_data(input_xlsx)

    print("Data composition per matrix (obs / censored / zero):")
    for m in MATRIX_ORDER:
        d = data[m]
        print(f"  {m:<12} obs={int((d.status=='obs').sum()):>3}  "
              f"cens={int((d.status=='cens').sum()):>3}  "
              f"zero={int((d.status=='zero').sum()):>3}  "
              f"farms={d.farm.nunique()}")
    print()

    mu_frames, gm_rows = [], []
    for matrix in MATRIX_ORDER:
        print(f"Fitting 3-level zero-inflated model for {matrix} ...")
        idata, farms = fit_full_model(data[matrix], matrix, draws, tune, chains)
        mu_df, gm = summarize(idata, farms, matrix)
        mu_frames.append(mu_df); gm_rows.append(gm)
        print(f"   eta = {gm['eta_log10_median']:.3f} log10 "
              f"[{gm['eta_log10_q025']:.3f}, {gm['eta_log10_q975']:.3f}] "
              f"{MATRIX_UNITS[matrix]}   R-hat={gm['max_rhat']}  "
              f"ESS={gm['min_ess']}  div={gm['divergences']}")

    mu_all = pd.concat(mu_frames, ignore_index=True)
    grand = pd.DataFrame(gm_rows)
    mu_all.to_csv(out / "bhm_full_mu_F.csv", index=False)
    grand.to_csv(out / "bhm_full_grand_means.csv", index=False)
    plot_forest(mu_all, grand, out / "fig_bhm_full_conc.png")

    print("\nGrand means (linear):")
    for _, r in grand.iterrows():
        print(f"  {r['matrix']:<12} {r['C_linear_median']:.2f} "
              f"[{r['C_linear_q025']:.2f}, {r['C_linear_q975']:.2f}] {r['unit']}")
    print(f"\nTotal posterior draws per matrix: {grand['total_draws'].iloc[0]}")
    print(f"Saved outputs in: {out.resolve()}")

# Define the parameters for the main function
input_xlsx_file = '/content/MRSA_sample_data.xlsx'
output_directory = './bhm_full_results'
draws = 2000
tune = 2000
chains = 4

# Call the main function to run the BHM model
main(input_xlsx_file, output_directory, draws, tune, chains)


Data composition per matrix (obs / censored / zero):
  Milk         obs= 12  cens=  1  zero=  5  farms=6
  Water        obs= 13  cens= 19  zero=  0  farms=9
  Bioaerosol   obs= 25  cens=  0  zero=  2  farms=8
  EnvDust      obs= 40  cens= 11  zero= 17  farms=10

Fitting 3-level zero-inflated model for Milk ...
   eta = 2.279 log10 [1.285, 3.288] CFU/mL   R-hat=1.0019  ESS=2326  div=0
Fitting 3-level zero-inflated model for Water ...
   eta = -0.183 log10 [-0.902, 0.276] CFU/mL   R-hat=1.0038  ESS=2422  div=0
Fitting 3-level zero-inflated model for Bioaerosol ...
   eta = 1.743 log10 [1.504, 2.026] CFU/m3   R-hat=1.002  ESS=3258  div=0
Fitting 3-level zero-inflated model for EnvDust ...
   eta = 0.743 log10 [0.176, 1.200] CFU/cm2   R-hat=1.0016  ESS=1478  div=0

Grand means (linear):
  Milk         190.19 [19.26, 1939.52] CFU/mL
  Water        0.66 [0.13, 1.89] CFU/mL
  Bioaerosol   55.32 [31.92, 106.23] CFU/m3
  EnvDust      5.53 [1.50, 15.84] CFU/cm2

Total posterior draws per matrix:

In [ ]:
# This cell previously attempted to execute another cell, which caused an error.
# Its content has been cleared as this functionality is not supported.

In [ ]:
# This cell previously attempted to execute another cell, which caused an error.
# Its content has been cleared as this functionality is not supported.

In [ ]:
pip install pymc arviz pytensor

## Run BHM Model

Now, let's run the Bayesian Hierarchical Model. We'll set the input and output paths, along with the sampling parameters (draws, tune, chains).

In [ ]:
# The logic for defining parameters and calling main has been moved to cell Jwk0DJDybSgS.